# Sampling Distributions

This notebook has the plots and code to support the sampling, bias and confidence intervals lecture.
It is adopted and adapted from Michael Ekstrand.

## Setup

Python modules:

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as sps
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

Set up a random number generator:

In [ ]:
rng = np.random.default_rng

Load the Penguin data:

In [ ]:
penguins = pd.read_csv('penguins.csv')
penguins.head()

And we're going to want to look at just the chinstraps sometimes:

In [ ]:
chinstraps = penguins[penguins['species'] == 'Chinstrap']
chinstraps.info()

That's all we need this time!

## Sampling Distributions

Let's say we have a population that is accurately described by the exponential distribution $\mathrm{Exp(1)}$ — with rate parameter $\lambda = 1$. This distribution has mean $\mu = 1$ and standard deviation $\sigma = 1$.

We can draw a sample of size 200 from this distribution:

In [ ]:
dist = sps.expon()

> **Note:** Here I choose to use the stats library in Scipy, which lets me create a distribution as an object. Below, we then use the `.rvs` method to sample from the distribution `dist`.

In [ ]:
sample = dist.rvs(size=200, random_state=rng)

And show its distribution:

In [ ]:
sns.distplot(sample,kde=False)

> **Note:** The curve in a Seaborn `dis(t)plot` is the *kernel density estimate*: it is an estimate of the probability density function. Note that it does not know that we are looking at a non-negative distribution (the exponential distribution can only generate values >= 0), so it's shape on the left side is off.

We can also compute statistics, such as the mean and standard deviation:

In [ ]:
np.mean(sample)

In [ ]:
np.std(sample,ddof=1)

That mean is pretty close to our true population mean of 1!  But it isn't quite exact - any given sample is going to have some error.  There is natural variation in samples that affects the statistic.

But what if we did this 1000 times? And each time, we took another sample of 200, and computed its mean?

How would this sample of sample means be distributed?

> **Note:** The syntax `[expr for var in iterable]` is a *list comprehension*. It lets is quickly write lists based on loops.

In [ ]:
sample_means = [np.mean(dist.rvs(size=200, random_state=rng)) for i in range(1000)]
sns.distplot(sample_means)
plt.xlabel('mean')

We call this distribution the **sampling distribution** of a statistic.

Let's overlay these on top of each other:

In [ ]:
plt.plot(np.linspace(0, 5), dist.pdf(np.linspace(0,5)), label='Population')
sns.distplot(sample, label='Sample')
sns.kdeplot(sample_means, label='Sampling Dist')

The sample mean is normally distributed with mean $\mu$ and standard deviation $\sigma/\sqrt{n}$.  Let's see that for a few sample sizes, with the true mean plotted as a vertical line:

In [ ]:
sample_means_100 = [np.mean(dist.rvs(size=100, random_state=rng)) for i in range(1000)]
sample_means_200 = [np.mean(dist.rvs(size=200, random_state=rng)) for i in range(1000)]
sample_means_500 = [np.mean(dist.rvs(size=500, random_state=rng)) for i in range(1000)]
sample_means_1000 = [np.mean(dist.rvs(size=1000, random_state=rng)) for i in range(1000)]

plt.axvline(1, color='black', linestyle=':')

sns.kdeplot(sample_means_100, label='n = 100')
sns.kdeplot(sample_means_200, label='n = 200')
sns.kdeplot(sample_means_500, label='n = 500')
sns.kdeplot(sample_means_1000, label='n = 1000')


plt.legend()
plt.show()

### Confidence Intervals

I'm going to write a little function for computing confidence intervals.  It will return a (low, high) tuple.

In [ ]:
def ci95(mean, sd, n, width=1.96):
    se = sd / np.sqrt(n)
    off = se * width
    return mean - off, mean + off

What's the estimated CI from our sample?

> **Note:** `ddof=1` tells NumPy's [`std`](https://numpy.org/doc/stable/reference/generated/numpy.std.html) function to compute the sample standard deviation.

In [ ]:
ci95(np.mean(sample), np.std(sample, ddof=1), 200)

Remember from lecture what the confidence interval really means and see [Confidence in Confidence](https://medium.com/@EpiEllie/having-confidence-in-confidence-intervals-8f881712d837).

In [ ]:
def samp_ci(size=200):
    sample = dist.rvs(size=size, random_state=rng)
    mean = np.mean(sample)
    std = np.std(sample, ddof=1)
    return ci95(mean, std, size)

We're going to use this function with another trick for building Pandas data frames — the `from_records` function, to build from an iterable of rows.  This will give us 1000 intervals:

In [ ]:
samp_cis = pd.DataFrame.from_records(
    [samp_ci() for i in range(1000)],
    columns=['lo', 'hi']
)
samp_cis

Mark the samples where it contains the true mean of 1, and see what fraction there are:

In [ ]:
samp_cis['hit'] = (samp_cis['lo'] <= 1) & (samp_cis['hi'] >= 1)
samp_cis.mean()

94.4% of our confindence intervals contain the mean. That is quite close to 95%.

> **Practice:** we can treat 'confidence interval contains the true mean' as an event with 95% probability of success.  What distribution describes the number of expected successess for 1000 trials at 95% success?  What is the probability that we would see a value in the range $950 \pm 10$ for such a set of trials?